# Unsupervised and Supervised learning analysis for AI Lab Gene Expression Data

This notebook will use machine learning methods to study the SmartSeq and DropSeq gene expression data.

The goal is not only to get high accuracy, but also to learn useful patterns from the data.

I will use in my part of research:
- KNN classifier
- MLP classifier
- K-means clustering

In [25]:
%pip install -r ../requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


### 0. Imports

In [26]:
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import random

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### 1. Paths

In [27]:
ROOT = Path(r"D:\Studi\Bocconi\AILab\ai-lab-gene-expression\data")

SMART = ROOT / "SmartSeq"
DROP = ROOT / "DropSeq"

assert SMART.exists() and DROP.exists(), f'Data folders not found, check {ROOT}'

OUTPUT = ROOT.parent / "nn_outputs"
OUTPUT.mkdir(exist_ok=True)

print(f'Notebook booted with data root={ROOT}')

Notebook booted with data root=D:\Studi\Bocconi\AILab\ai-lab-gene-expression\data


### 2. Helper functions for loading the data

In [28]:
def drop_quotes(df):
    return df.astype(str).str.replace('"', '', regex=False)

def load_expr(path):
    df = pd.read_csv(path, sep=r"\s+", engine="python", index_col=0)
    df.index = drop_quotes(df.index)
    df.columns = drop_quotes(df.columns)
    return df

def get_dropseq_label(sample_name):
    if "Normo" in sample_name:
        return "Normoxia"
    if "Hypo" in sample_name:
        return "Hypoxia"
    return "Unknown"

### 3. Load normalized training data

In [29]:
smart_mcf7 = load_expr(SMART / "MCF7_SmartS_Filtered_Normalised_3000_Data_train.txt")
smart_hcc1806 = load_expr(SMART / "HCC1806_SmartS_Filtered_Normalised_3000_Data_train.txt")

drop_mcf7 = load_expr(DROP / "MCF7_Filtered_Normalised_3000_Data_train.txt")
drop_hcc1806 = load_expr(DROP / "HCC1806_Filtered_Normalised_3000_Data_train.txt")

meta_mcf7 = pd.read_csv(SMART / "MCF7_SmartS_MetaData.tsv", sep="\t")
meta_hcc1806 = pd.read_csv(SMART / "HCC1806_SmartS_MetaData.tsv", sep="\t")

meta_mcf7["Filename"] = drop_quotes(meta_mcf7["Filename"])
meta_hcc1806["Filename"] = drop_quotes(meta_hcc1806["Filename"].astype(str).str.replace('"', '', regex=False))

smart_mcf7_labels = meta_mcf7.set_index("Filename").loc[smart_mcf7.columns, "Condition"]
smart_hcc_labels = meta_hcc1806.set_index("Filename").loc[smart_hcc1806.columns, "Condition"]

drop_mcf7_labels = pd.Series([get_dropseq_label(c) for c in drop_mcf7.columns], index=drop_mcf7.columns)
drop_hcc_labels = pd.Series([get_dropseq_label(c) for c in drop_hcc1806.columns], index=drop_hcc1806.columns)

datasets = {
    "SmartSeq_MCF7": (smart_mcf7, smart_mcf7_labels),
    "SmartSeq_HCC1806": (smart_hcc1806, smart_hcc_labels),
    "DropSeq_MCF7": (drop_mcf7, drop_mcf7_labels),
    "DropSeq_HCC1806": (drop_hcc1806, drop_hcc_labels),
}

for name, (expr, labels) in datasets.items():
    print(name, expr.shape, labels.value_counts().to_dict())

SmartSeq_MCF7 (3000, 250) {'Norm': 126, 'Hypo': 124}
SmartSeq_HCC1806 (3000, 182) {'Hypo': 97, 'Normo': 85}
DropSeq_MCF7 (3000, 21626) {'Normoxia': 12705, 'Hypoxia': 8921}
DropSeq_HCC1806 (3000, 14682) {'Hypoxia': 8899, 'Normoxia': 5783}


In [30]:
class CrossValidation:
    def __init__(self, expr, labels):
        self.X, self.y = CrossValidation._prepare_xy(expr, labels)

        self.n_splits = 5
        self.batch_size = 32

        self.skf = StratifiedKFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=SEED
        )

    def _to_binary_labels(labels):
        return labels.astype(str).str.contains("Hyp", case=False, na=False).astype(int)

    def _prepare_xy(expr, labels):
        X = expr.T.copy()
        y = np.array(CrossValidation._to_binary_labels(labels.loc[X.index]))
        return X, y

    def get_loaders(self):
        for fold, (train_idx, val_idx) in enumerate(self.skf.split(self.X, self.y)):

            # Scale using training data only
            scaler = StandardScaler()

            X_train = scaler.fit_transform(self.X.iloc[train_idx])
            X_val = scaler.transform(self.X.iloc[val_idx])

            y_train = self.y[train_idx]
            y_val = self.y[val_idx]

            # Convert to tensors
            X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train, dtype=torch.long)

            X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
            y_val_tensor = torch.tensor(y_val, dtype=torch.long)

            # Create datasets
            train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
            val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

            # Create dataloaders
            train_loader = DataLoader(
                train_dataset,
                batch_size=self.batch_size,
                shuffle=True
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=self.batch_size,
                shuffle=False
            )
            
            yield train_loader, val_loader, scaler, X_train, X_val, y_train, y_val

In [31]:
def train_knn(DATASET_NAME, n_neighbors=3, metric="euclidean", weights="uniform", use_pca=True, n_components=50):
    expr, labels = datasets[DATASET_NAME]
    cv = CrossValidation(expr, labels)

    results = []

    for fold, (train_loader, val_loader, scaler, X_train, X_val, y_train, y_val) in enumerate(cv.get_loaders()):
        print(f"\nFold {fold + 1}")
        
        if use_pca:
            pca = PCA(n_components=n_components, random_state=SEED)

            X_train = pca.fit_transform(X_train)
            X_val = pca.transform(X_val)

        model = KNeighborsClassifier(
            n_neighbors=n_neighbors,
            metric=metric,
            weights=weights
        )

        # KNN does not use epochs or batches
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)[:, 1]

        acc = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)
        report = classification_report(y_val, y_pred, zero_division=0)

        print("Accuracy:", acc)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1:", f1)
        print("Confusion matrix:")
        print(cm)

        results.append({
            "fold": fold + 1,
            "model": model,
            "scaler": scaler,

            "accuracy_score": acc,
            "precision_score": precision,
            "recall_score": recall,
            "f1_score": f1,
            "confusion_matrix": cm,
            "classification_report": report,

            "X_train": X_train,
            "X_val": X_val,
            "y_train": y_train,
            "y_val": y_val,
            "y_pred": y_pred,
            "y_prob": y_prob
        })

    return results

In [32]:
knn_results = {}
for key in datasets:
    knn_results[key] = train_knn(key)


Fold 1
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[26  0]
 [ 0 24]]

Fold 2
Accuracy: 0.98
Precision: 0.9615384615384616
Recall: 1.0
F1: 0.9803921568627451
Confusion matrix:
[[24  1]
 [ 0 25]]

Fold 3
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 4
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 5
Accuracy: 0.96
Precision: 0.96
Recall: 0.96
F1: 0.96
Confusion matrix:
[[24  1]
 [ 1 24]]

Fold 1
Accuracy: 0.918918918918919
Precision: 0.9473684210526315
Recall: 0.9
F1: 0.9230769230769231
Confusion matrix:
[[16  1]
 [ 2 18]]

Fold 2
Accuracy: 0.918918918918919
Precision: 1.0
Recall: 0.85
F1: 0.918918918918919
Confusion matrix:
[[17  0]
 [ 3 17]]

Fold 3
Accuracy: 0.9444444444444444
Precision: 0.9473684210526315
Recall: 0.9473684210526315
F1: 0.9473684210526315
Confusion matrix:
[[16  1]
 [ 1 18]]

Fold 4
Accuracy: 0.8611111111111112
Precision: 1.0
Recall: 0.7368421052631579
F

In [34]:
#class_names = ["Normoxia", "Hypoxia"]

#for key, results in knn_results.items():
#    cms = results["confusion_matrix"]
#
#    mean_cm = cms.mean(axis=0)
#    std_cm = cms.std(axis=0)

#    plt.figure(figsize=(5, 4))

#    labels = np.array([
#        [f"{mean_cm[i, j]:.1f}\n±{std_cm[i, j]:.1f}" for j in range(mean_cm.shape[1])]
#        for i in range(mean_cm.shape[0])
#    ])

#    sns.heatmap(
#        mean_cm,
#        annot=labels,
#        fmt="",
#        cmap="Blues",
#        xticklabels=class_names,
#        yticklabels=class_names
#    )

#    plt.title(f"Mean Confusion Matrix Across Folds - {key}")
#    plt.xlabel("Predicted")
#    plt.ylabel("Actual")
#    plt.tight_layout()
#    plt.show()

    

In [36]:
import matplotlib.pyplot as plt
import numpy as np

k_values = range(1, 21)

for key in datasets:
    if "Drop" in key:
        continue

    plt.figure(figsize=(8, 5))
    scores = []

    for k in k_values:
        acc = train_knn(key, k)["accuracy_score"]
        scores.append(acc)

    scores = np.mean(scores, axis=1)

    print(scores)

    plt.plot(
        k_values,
        scores,
        marker='o',
        linewidth=3,
        label="Mean Accuracy"
    )

    best_k = k_values[np.argmax(scores)]
    best_acc = np.max(scores)

    plt.axvline(best_k, linestyle="--", alpha=0.6)

    plt.title(f"KNN Accuracy vs K - {key}")
    plt.xlabel("K")
    plt.ylabel("Accuracy")
    plt.xticks(k_values)
    plt.legend()
    plt.grid(True)

    print(f"{key} -> Best K = {best_k}, Accuracy = {best_acc:.4f}")

    plt.show()


Fold 1
Accuracy: 0.98
Precision: 0.96
Recall: 1.0
F1: 0.9795918367346939
Confusion matrix:
[[25  1]
 [ 0 24]]

Fold 2
Accuracy: 0.96
Precision: 0.96
Recall: 0.96
F1: 0.96
Confusion matrix:
[[24  1]
 [ 1 24]]

Fold 3
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 4
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 5
Accuracy: 0.98
Precision: 0.9615384615384616
Recall: 1.0
F1: 0.9803921568627451
Confusion matrix:
[[24  1]
 [ 0 25]]


TypeError: list indices must be integers or slices, not str

<Figure size 800x500 with 0 Axes>

Notes: learning knn on large datasets such as dropseq and on relatively big k is a mess! include this in the report

In [37]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [38]:
def train_mlp(DATASET_NAME, epochs=30, lr=1e-3, use_pca=True, n_components=50):
    expr, labels = datasets[DATASET_NAME]
    cv = CrossValidation(expr, labels)
    input_dim = cv.X.shape[1]

    results = []

    for fold, (train_loader, val_loader, scaler, X_train, X_val, y_train, y_val) in enumerate(cv.get_loaders()):
        print(f"\nFold {fold + 1}")

        if use_pca:
            pca = PCA(n_components=n_components, random_state=SEED)

            X_train = pca.fit_transform(X_train)
            X_val = pca.transform(X_val)

        model = MLPClassifier(input_dim).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # ------------------
            # Training
            # ------------------
            model.train()
            running_train_loss = 0.0

            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(DEVICE)

                # BCEWithLogitsLoss expects float labels shaped (batch, 1)
                batch_y = batch_y.float().view(-1, 1).to(DEVICE)

                optimizer.zero_grad()

                logits = model(batch_X)
                loss = criterion(logits, batch_y)

                loss.backward()
                optimizer.step()

                running_train_loss += loss.item()

            avg_train_loss = running_train_loss / len(train_loader)
            train_losses.append(avg_train_loss)

            # ------------------
            # Validation loss
            # ------------------
            model.eval()
            running_val_loss = 0.0

            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X = batch_X.to(DEVICE)
                    batch_y = batch_y.float().view(-1, 1).to(DEVICE)

                    logits = model(batch_X)
                    loss = criterion(logits, batch_y)

                    running_val_loss += loss.item()

            avg_val_loss = running_val_loss / len(val_loader)
            val_losses.append(avg_val_loss)

            if (epoch + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Train Loss: {avg_train_loss:.4f} | "
                    f"Val Loss: {avg_val_loss:.4f}"
                )

        # ------------------
        # Final validation predictions
        # ------------------
        model.eval()

        y_probs = []
        y_preds = []

        with torch.no_grad():
            for batch_X, _ in val_loader:
                batch_X = batch_X.to(DEVICE)

                logits = model(batch_X)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).long()

                y_probs.extend(probs.cpu().numpy().flatten())
                y_preds.extend(preds.cpu().numpy().flatten())

        y_probs = np.array(y_probs)
        y_pred = np.array(y_preds)

        acc = accuracy_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred, zero_division=0)
        recall = recall_score(y_val, y_pred, zero_division=0)
        f1 = f1_score(y_val, y_pred, zero_division=0)
        cm = confusion_matrix(y_val, y_pred)
        report = classification_report(y_val, y_pred, zero_division=0)

        print("Accuracy:", acc)
        print("Precision:", precision)
        print("Recall:", recall)
        print("F1:", f1)
        print("Confusion matrix:")
        print(cm)

        results.append({
            "fold": fold + 1,
            "model": model,
            "scaler": scaler,

            "accuracy_score": acc,
            "precision_score": precision,
            "recall_score": recall,
            "f1_score": f1,
            "confusion_matrix": cm,
            "classification_report": report,

            "train_losses": train_losses,
            "val_losses": val_losses,

            "X_train": X_train,
            "X_val": X_val,
            "y_train": y_train,
            "y_val": y_val,
            "y_pred": y_pred,
            "y_prob": y_probs
        })

    return results

In [ ]:
mlp_results = {}

for key in datasets:
    mlp_results[key] = train_mlp(key)


Fold 1
Epoch 10/30 | Train Loss: 0.0006 | Val Loss: 0.0016
Epoch 20/30 | Train Loss: 0.0003 | Val Loss: 0.0012
Epoch 30/30 | Train Loss: 0.0001 | Val Loss: 0.0010
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[26  0]
 [ 0 24]]

Fold 2
Epoch 10/30 | Train Loss: 0.0001 | Val Loss: 0.0181
Epoch 20/30 | Train Loss: 0.0001 | Val Loss: 0.0167
Epoch 30/30 | Train Loss: 0.0000 | Val Loss: 0.0157
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 3
Epoch 10/30 | Train Loss: 0.0002 | Val Loss: 0.0013
Epoch 20/30 | Train Loss: 0.0002 | Val Loss: 0.0008
Epoch 30/30 | Train Loss: 0.0001 | Val Loss: 0.0005
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]

Fold 4
Epoch 10/30 | Train Loss: 0.0006 | Val Loss: 0.0088
Epoch 20/30 | Train Loss: 0.0001 | Val Loss: 0.0073
Epoch 30/30 | Train Loss: 0.0002 | Val Loss: 0.0066
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
Confusion matrix:
[[25  0]
 [ 0 25]]


In [ ]:
#for key, results in mlp_results.items():
#    accuracies = [r["accuracy_score"] for r in results]
#    f1s = [r["f1_score"] for r in results]

#    print(f"\n{key}")
#    print("Mean accuracy:", np.mean(accuracies))
#    print("Std accuracy:", np.std(accuracies))
#    print("Mean F1:", np.mean(f1s))
#    print("Std F1:", np.std(f1s))